# 🚀 Kaggle Notebook Dashboard – One-Click Deploy

Run all cells top-to-bottom. Your app will be live in ~3 minutes.

You only need to fill in **Cell 2** with your credentials.

> ⚠️ Colab sessions last ~12 hours. For 24/7 hosting use Docker or HuggingFace Spaces.

In [ ]:
# ── Cell 1: Install system dependencies ──────────────────────────────────────
!apt-get install -y nodejs npm git > /dev/null 2>&1
!npm install -g pnpm@latest > /dev/null 2>&1
!pip install -q pyngrok requests
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Your credentials ─────────────────────────────────────────────────
# Fill these in, then run this cell

KAGGLE_USERNAME    = 'your_kaggle_username'    # e.g. wadud919249
KAGGLE_API_KEY     = 'your_kaggle_api_key'     # from kaggle.com → Account → API
TELEGRAM_BOT_TOKEN = 'your_telegram_bot_token' # from @BotFather on Telegram
NGROK_AUTH_TOKEN   = ''  # optional but recommended – free at ngrok.com/signup

print('✅ Credentials set')

In [ ]:
# ── Cell 3: Clone the repo and install packages ───────────────────────────────
import os
if not os.path.exists('/content/app'):
    !git clone https://github.com/shan-test-project/tg-bot-to-kaggle.git /content/app
else:
    !git -C /content/app pull
%cd /content/app
!pnpm install --frozen-lockfile 2>&1 | tail -3
print('✅ Code ready')

In [ ]:
# ── Cell 4: Build the app ────────────────────────────────────────────────────
!pnpm --filter @workspace/kaggle-dashboard run build 2>&1 | tail -3
!pnpm --filter @workspace/api-server run build 2>&1 | tail -3
print('✅ Build complete')

In [ ]:
# ── Cell 5: Start server + expose via ngrok + configure settings ──────────────
import subprocess, time, requests as req, os
from pyngrok import ngrok

PORT = 8080
os.makedirs('/content/app/data', exist_ok=True)

# Start the server
proc = subprocess.Popen(
    ['node', '--enable-source-maps', 'artifacts/api-server/dist/index.mjs'],
    cwd='/content/app',
    env={**os.environ, 'PORT': str(PORT), 'NODE_ENV': 'production', 'DATA_DIR': '/content/app/data'}
)

# Wait for server to be ready
for i in range(30):
    time.sleep(1)
    try:
        r = req.get(f'http://localhost:{PORT}/api/healthz', timeout=2)
        if r.status_code == 200:
            print(f'✅ Server running on port {PORT}')
            break
    except: pass
else:
    print('❌ Server failed to start')

# Open ngrok tunnel
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url
print(f'🌍 Public URL: {PUBLIC_URL}')

# Save credentials via the settings API
resp = req.put(f'http://localhost:{PORT}/api/settings', json={
    'kaggleUsername': KAGGLE_USERNAME,
    'kaggleKey': KAGGLE_API_KEY,
    'telegramBotToken': TELEGRAM_BOT_TOKEN,
})
print('⚙️  Settings saved:', resp.json())

# Register the Telegram webhook
resp2 = req.post(f'{PUBLIC_URL}/api/telegram/setup')
print('🤖 Bot webhook:', resp2.json())

print(f'\n✅ ALL DONE!')
print(f'👉 Open your Telegram bot and type /start')
print(f'🌐 Dashboard URL: {PUBLIC_URL}')